# Trading Journal

This notebook has two independent parts that share the same Airtable connection.

## What we are building

A systematic trader generates two kinds of records every day: **trades** and **thoughts**.
Most traders capture trades (because their broker forces them to) but ignore thoughts
(because nothing forces them to). This notebook captures both in Airtable so they can
be analysed together.

**Part A — Trade Reconciliation** runs after the market closes.
It compares what you *ordered* against what actually *executed*, classifies every
discrepancy, and updates a live portfolio CSV. The insight comes from the three outcomes:

| Outcome | Meaning |
|---|---|
| matched | order filled — normal trade |
| missed | order placed, never filled — what you *intended* but not executed |
| override | filled without an order — reactive, unplanned trade |

Missed trades are the most valuable signal. Abraham Wald showed in WW2 that the planes
that never came back carried the most important information. Missed trades are your
planes that never came back.

**Part B — Psychology Journal Analysis** runs weekly (or on demand).
Daily entries are submitted via two Airtable form links (no notebook needed for that).
This part loads all entries, merges morning and evening rows by date, computes
a gamified score and streak, and patches the results back to Airtable.

## Prerequisites

1. A `.env` file with `AIRTABLE_API_KEY`, `AIRTABLE_BASE_ID`, `AIRTABLE_TABLE`
2. Both Airtable tables created by running `2026-04-23-airtable-setup.ipynb` first
3. For Part B: at least one Morning Check-in and one Evening Review form submission

## Workflow

```
Daily (after close)  : run Part A  — reconcile orders vs executions
Morning (phone)      : open Morning Check-in form link — 2 minutes
Evening (phone)      : open Evening Review form link   — 5 minutes
Weekly (at desk)     : run Part B  — merge, score, review
```

---
# Part A · Trade Reconciliation

**Steps:**
1. Load credentials and build the Airtable connection
2. Define three low-level Airtable helpers (list, append, patch)
3. Define a CSV loader with a safe fallback for missing files
4. Provide sample data covering all three reconciliation cases
5. Seed pre-existing positions into the TradeLog (run once)
6. Split today's orders vs executions into matched / missed / override
7. Write missed and override records to the TradeLog
8. Route each matched trade to the right position handler
9. Orchestrate the full reconciliation in one function
10. Run, save the updated portfolio CSV, display summary

## 1 · Setup

Load the three required environment variables from `.env`.
Using `os.environ.setdefault` means variables already set in the shell
are never overwritten — safe to run in any environment.

Required keys: `AIRTABLE_API_KEY`, `AIRTABLE_BASE_ID`, `AIRTABLE_TABLE`.

In [ ]:
import os, requests, pandas as pd
from pathlib import Path
from datetime import date, timedelta

def load_env(path='.env'):
    if not Path(path).exists():
        raise FileNotFoundError('No .env -- copy .env.example and fill in credentials.')
    with Path(path).open() as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#') or '=' not in line: continue
            k, _, v = line.partition('=')
            os.environ.setdefault(k.strip(), v.strip())

load_env()
missing = [k for k in ('AIRTABLE_API_KEY','AIRTABLE_BASE_ID','AIRTABLE_TABLE')
           if not os.getenv(k)]
if missing: raise EnvironmentError(f'Missing in .env: {missing}')
print('credentials OK')

## 2 · Airtable helpers

All Airtable communication flows through three functions:

- **`at_list_open(ticker, strategy)`** — fetches open lots for a position,
  sorted oldest-first. Used by `_close_fifo` to know which lots to close.
- **`at_append(fields, url)`** — POSTs a new record. Strips `None` and `NaN`
  values before sending so Airtable never receives invalid nulls.
- **`at_patch(record_id, fields, url)`** — PATCHes an existing record.
  Used to fill exit fields when a position closes.

Both `at_append` and `at_patch` accept an optional `url` parameter that
defaults to the TradeLog endpoint — Part B passes `AT_J_URL` to reuse them
for the journal table.

In [ ]:
BASE_ID    = os.getenv('AIRTABLE_BASE_ID')
AT_URL     = f"https://api.airtable.com/v0/{BASE_ID}/{os.getenv('AIRTABLE_TABLE')}"
AT_J_URL   = f'https://api.airtable.com/v0/{BASE_ID}/journal'
AT_HEADERS = {'Authorization': f"Bearer {os.getenv('AIRTABLE_API_KEY')}",
              'Content-Type': 'application/json'}

PORTFOLIO_COLS = ['ticker','position','cost','stop_price','target_price','strategy','entry_date']

def at_list_open(ticker, strategy):
    formula = f"AND({{ticker}}='{ticker}', {{strategy}}='{strategy}', NOT({{exit_date}}))"
    r = requests.get(AT_URL, headers=AT_HEADERS,
                     params={'filterByFormula': formula,
                             'sort[0][field]': 'entry_date',
                             'sort[0][direction]': 'asc'})
    r.raise_for_status()
    return r.json().get('records', [])

def at_append(fields, url=None):
    url   = url or AT_URL
    clean = {k: v for k, v in fields.items()
             if v is not None and not (isinstance(v, float) and pd.isna(v))}
    r = requests.post(url, headers=AT_HEADERS, json={'fields': clean})
    r.raise_for_status()
    return r.json()

def at_patch(record_id, fields, url=None):
    url = url or AT_URL
    r   = requests.patch(f'{url}/{record_id}', headers=AT_HEADERS, json={'fields': fields})
    r.raise_for_status()
    return r.json()

## 3 · load_csv

A one-liner that returns an empty DataFrame when the file does not exist
instead of raising an error. This lets the notebook run cleanly on day one
before any CSV has been created.

In [ ]:
def load_csv(path):
    p = Path(path)
    return pd.read_csv(p) if p.exists() else pd.DataFrame()

## 4 · Sample data

Six DataFrames that cover every reconciliation case in a single run.
All tickers and brokers are fictitious — for book illustration only.

**Brokers:** IronBridge Capital · NexusTrade Securities

| DataFrame | Purpose |
|---|---|
| `sample_portfolio` | open positions going into today — some built over multiple lots |
| `sample_orders` | what you sent to your brokers today |
| `sample_executions` | what the brokers actually filled today |

Six scenarios covered:

| # | Case | Ticker | Description |
|---|---|---|---|
| 1 | seed / build | KROX | short pyramid started three weeks ago; today adds a 4th tranche |
| 2 | full close | VEGA | short entered 3 weeks ago, exited in full today (win) |
| 3 | partial exit | PULS | short seeded as two lots; FIFO closes the oldest lot only |
| 4 | reversal | FLUX | short -150 → buy +300 → closes the short, opens a new long |
| 5 | missed | NOVA | order placed, never filled — the Wald signal |
| 6 | override | ORION | fill with no prior order — unplanned, reactive trade |

> **Note on PULS:** the position is split across two portfolio rows (two add entries).
> `_seed_portfolio` creates two separate Airtable records so `_close_fifo` can consume
> them oldest-first — a clean FIFO demonstration.

In [ ]:
# ── Portfolio going into today ────────────────────────────────────────────
# KROX short was built as a pyramid: 3 tranches over 3 weeks
#   Lot 1  2026-03-31  -100 @ 57.00
#   Lot 2  2026-04-07  -100 @ 55.80  <- broker: IronBridge
#   Lot 3  2026-04-14  -100 @ 54.30  avg cost = 55.70
# Seeded as three separate rows so FIFO works correctly.
sample_portfolio = pd.DataFrame([
    # KROX: pyramid short — three lots, seeded as individual entries
    {'ticker':'KROX','position':-100,'cost':57.00,'stop_price':61.00,
     'target_price':47.00,'strategy':'momentum','entry_date':'2026-03-31'},
    {'ticker':'KROX','position':-100,'cost':55.80,'stop_price':61.00,
     'target_price':47.00,'strategy':'momentum','entry_date':'2026-04-07'},
    {'ticker':'KROX','position':-100,'cost':54.30,'stop_price':61.00,
     'target_price':47.00,'strategy':'momentum','entry_date':'2026-04-14'},
    # VEGA: single short — full close today
    {'ticker':'VEGA','position':-200,'cost':42.50,'stop_price':46.00,
     'target_price':36.00,'strategy':'momentum','entry_date':'2026-04-01'},
    # PULS: two-lot pyramid — partial (FIFO) exit today closes oldest lot only
    {'ticker':'PULS','position':-200,'cost':29.10,'stop_price':32.00,
     'target_price':22.00,'strategy':'trend','entry_date':'2026-04-08'},
    {'ticker':'PULS','position':-200,'cost':28.40,'stop_price':32.00,
     'target_price':22.00,'strategy':'trend','entry_date':'2026-04-10'},
    # FLUX: single short — reversal to long today
    {'ticker':'FLUX','position':-150,'cost':18.40,'stop_price':21.00,
     'target_price':13.00,'strategy':'momentum','entry_date':'2026-04-15'},
])

# ── Today's orders (2026-04-22) ────────────────────────────────────────────
sample_orders = pd.DataFrame([
    # 1. Add to KROX short (4th tranche of pyramid)
    {'date':'2026-04-22','ticker':'KROX','broker':'IronBridge','quantity':-100,
     'price':53.50,'order_type':'limit','stop_price':61.00,
     'target_price':45.00,'risk_pct':0.5,'strategy':'momentum'},
    # 2. Full close VEGA short
    {'date':'2026-04-22','ticker':'VEGA','broker':'IronBridge','quantity':200,
     'price':40.80,'order_type':'limit','stop_price':None,
     'target_price':None,'risk_pct':None,'strategy':'momentum'},
    # 3. Partial exit PULS (closes oldest lot only — FIFO)
    {'date':'2026-04-22','ticker':'PULS','broker':'NexusTrade','quantity':200,
     'price':27.50,'order_type':'limit','stop_price':None,
     'target_price':None,'risk_pct':None,'strategy':'trend'},
    # 4. Reversal FLUX: buy 300 to close -150 short and open +150 long
    {'date':'2026-04-22','ticker':'FLUX','broker':'NexusTrade','quantity':300,
     'price':19.20,'order_type':'stop','stop_price':17.50,
     'target_price':24.00,'risk_pct':1.0,'strategy':'momentum'},
    # 5. NOVA — missed trade (Wald signal: intent without execution)
    {'date':'2026-04-22','ticker':'NOVA','broker':'IronBridge','quantity':-50,
     'price':22.00,'order_type':'limit','stop_price':24.50,
     'target_price':17.00,'risk_pct':0.5,'strategy':'trend'},
])

# ── Today's executions (2026-04-22) ────────────────────────────────────────
sample_executions = pd.DataFrame([
    # 1. KROX add — slight positive slippage
    {'date':'2026-04-22','ticker':'KROX','broker':'IronBridge','quantity_exec':-100,
     'price_exec':53.48,'commission':2.50,'execution_id':'EX001'},
    # 2. VEGA full close — win
    {'date':'2026-04-22','ticker':'VEGA','broker':'IronBridge','quantity_exec':200,
     'price_exec':40.82,'commission':5.00,'execution_id':'EX002'},
    # 3. PULS partial exit — closes oldest lot
    {'date':'2026-04-22','ticker':'PULS','broker':'NexusTrade','quantity_exec':200,
     'price_exec':27.52,'commission':5.00,'execution_id':'EX003'},
    # 4. FLUX reversal — close 150 short, open 150 long in one execution
    {'date':'2026-04-22','ticker':'FLUX','broker':'NexusTrade','quantity_exec':300,
     'price_exec':19.18,'commission':7.50,'execution_id':'EX004'},
    # 6. ORION — override (unplanned short, no prior order)
    {'date':'2026-04-22','ticker':'ORION','broker':'NexusTrade','quantity_exec':-75,
     'price_exec':31.50,'commission':1.88,'execution_id':'EX005'},
])

print(f'Portfolio: {len(sample_portfolio)} lot rows  |  '
      f'Orders: {len(sample_orders)}  |  '
      f'Executions: {len(sample_executions)}')

## 5 · _seed_portfolio

**The cold-start problem.** When you first set up the TradeLog your Airtable
is empty, but your portfolio is not. If you immediately start reconciling,
`_close_fifo` will find no open lots to close against.

`_seed_portfolio` solves this by writing every row in the portfolio as an
opening entry in the TradeLog. It checks `at_list_open` first for each
`(ticker, strategy)` pair; if ANY open lot already exists for that pair it
skips the entire ticker — so the function is safe to call multiple times.

**One portfolio row = one Airtable lot.** When a position was built across
multiple adds (like KROX with three tranches), pass three separate portfolio
rows — each with its own `entry_date` and `cost`. `_close_fifo` will then
consume them oldest-first, giving a faithful FIFO history.

> **Run once** after initial setup, then comment out this call.

In [ ]:
def _seed_portfolio(portfolio):
    for _, row in portfolio.iterrows():
        ticker, strategy = row['ticker'], row['strategy']
        if at_list_open(ticker, strategy):
            print(f'  SKIP   {ticker}/{strategy}  already in TradeLog')
            continue
        at_append({
            'entry_date':    row['entry_date'],
            'ticker':        ticker,
            'strategy':      strategy,
            'quantity_exec': int(row['position']),
            'price_exec':    round(float(row['cost']), 4),
            'stop_price':    row.get('stop_price'),
            'target_price':  row.get('target_price'),
        })
        print(f'  SEEDED {ticker}/{strategy}  pos={int(row["position"]):+d} @ {float(row["cost"]):.4f}')

# Run once to initialise -- comment out after first use
_seed_portfolio(sample_portfolio)

## 6 · _split_trades

**The core classification step.** A single `pd.merge` with `how='outer'`
and `indicator=True` replaces what would otherwise be three separate loops.

The merge key is `(date, ticker, broker)`. Every row in the merged result
gets a `_merge` label:

| `_merge` value | Meaning | Action |
|---|---|---|
| `both` | order AND execution found | reconcile into portfolio |
| `left_only` | order only, no fill | log as missed trade |
| `right_only` | fill only, no order | log as override |

The `_merge` column is dropped from all three outputs before returning
so downstream functions receive clean DataFrames.

In [ ]:
def _split_trades(orders, executions):
    merged = pd.merge(
        orders, executions,
        on=['date','ticker','broker'],
        how='outer', suffixes=('_ord','_exec'), indicator=True
    )
    matched  = merged[merged['_merge'] == 'both'].copy()
    missed   = merged[merged['_merge'] == 'left_only'].copy()
    override = merged[merged['_merge'] == 'right_only'].copy()
    return (matched.drop(columns='_merge'),
            missed.drop(columns='_merge'),
            override.drop(columns='_merge'))

## 7 · _write_missed and _write_override

These two functions handle the exception cases. Neither modifies the portfolio
CSV — they only write an audit record to the TradeLog.

**`_write_missed`** logs orders that were never filled. The `missed_trade`
field is set to `'missed'`. Think of these as your *intended* trades —
the systematic signals your process generated but the market rejected.
Studying them reveals your edge without survivorship bias.

**`_write_override`** logs fills that arrived without a prior order.
These are reactive or manual trades. The `missed_trade` field is set to
`'override'` to flag them for review.

In [ ]:
def _write_missed(missed):
    for _, row in missed.iterrows():
        at_append({
            'entry_date':     row['date'],
            'ticker':         row['ticker'],
            'strategy':       row.get('strategy'),
            'broker':         row['broker'],
            'order_type':     row.get('order_type'),
            'quantity_order': int(row['quantity']),
            'price_order':    row.get('price'),
            'stop_price':     row.get('stop_price'),
            'target_price':   row.get('target_price'),
            'risk_pct':       row.get('risk_pct'),
            'missed_trade':   'missed',
        })
        print(f"  MISSED   {row['ticker']}  qty={int(row['quantity']):+d}")

def _write_override(override):
    for _, row in override.iterrows():
        at_append({
            'entry_date':    row['date'],
            'ticker':        row['ticker'],
            'broker':        row['broker'],
            'quantity_exec': int(row['quantity_exec']),
            'price_exec':    float(row['price_exec']),
            'commission':    row.get('commission'),
            'missed_trade':  'override',
            'record_id':     row.get('execution_id'),
        })
        print(f"  OVERRIDE {row['ticker']}  qty={int(row['quantity_exec']):+d}")

## 8 · Position handlers

Four outcomes, four code paths. Each writes a TradeLog record and returns
an updated portfolio DataFrame.

**`_open_new`** — no existing position for this `(ticker, strategy)` pair.
Creates a new row in both the TradeLog and the portfolio CSV.

**`_add_lot`** — position already exists and the new trade is in the same
direction. Updates the portfolio row with a weighted average cost:
`(old_cost * |old_qty| + new_price * |new_qty|) / |total_qty|`.

**`_close_fifo`** — trade is in the opposite direction (closing or reducing).
Fetches open lots from Airtable sorted oldest-first (FIFO), then consumes
them in sequence until `to_close` shares are accounted for. Each lot gets
its exit fields patched: `exit_date`, `price_exit`, `gross_pnl`, `net_pnl`,
`trade_result`. Commission is allocated proportionally across lots.

**Reversal** (handled inside `_close_fifo`) — when `to_close` exceeds the
total open position, all existing lots are closed and the surplus opens a new
position in the opposite direction. Example: short -150, buy +300 → closes
the -150 short, appends a new +150 long lot, updates portfolio accordingly.

**Partial exit** (also inside `_close_fifo`) — when `to_close` is smaller
than the position, only the oldest lots are consumed (FIFO). The portfolio
row is updated to reflect the remaining position size.

In [ ]:
def _open_new(row, port):
    at_append({
        'entry_date':     row['date'],      'ticker':       row['ticker'],
        'strategy':       row['strategy'],  'broker':       row['broker'],
        'order_type':     row.get('order_type_ord') or row.get('order_type'),
        'quantity_order': int(row['quantity']),
        'price_order':    row.get('price'),  'stop_price':   row.get('stop_price'),
        'target_price':   row.get('target_price'), 'risk_pct': row.get('risk_pct'),
        'quantity_exec':  int(row['quantity_exec']),
        'price_exec':     float(row['price_exec']),
        'commission':     row.get('commission'),
        'record_id':      row.get('execution_id'),
    })
    new_row = pd.DataFrame([{
        'ticker':       row['ticker'],   'strategy':   row['strategy'],
        'position':     int(row['quantity_exec']),
        'cost':         round(float(row['price_exec']), 4),
        'stop_price':   row.get('stop_price'),
        'target_price': row.get('target_price'),
        'entry_date':   row['date'],
    }])
    print(f"  OPEN     {row['ticker']}/{row['strategy']}  "
          f"qty={int(row['quantity_exec']):+d} @ {float(row['price_exec']):.4f}")
    return pd.concat([port, new_row], ignore_index=True)


def _add_lot(row, port):
    at_append({
        'entry_date':     row['date'],     'ticker':      row['ticker'],
        'strategy':       row['strategy'], 'broker':      row['broker'],
        'order_type':     row.get('order_type_ord') or row.get('order_type'),
        'quantity_order': int(row['quantity']),
        'price_order':    row.get('price'),
        'quantity_exec':  int(row['quantity_exec']),
        'price_exec':     float(row['price_exec']),
        'commission':     row.get('commission'),
        'record_id':      row.get('execution_id'),
    })
    mask     = (port['ticker'] == row['ticker']) & (port['strategy'] == row['strategy'])
    old_pos  = int(port.loc[mask,'position'].values[0])
    old_cost = float(port.loc[mask,'cost'].values[0])
    qty      = int(row['quantity_exec'])
    new_pos  = old_pos + qty
    new_cost = round(
        (old_cost*abs(old_pos) + float(row['price_exec'])*abs(qty)) / abs(new_pos), 4)
    port.loc[mask,'position'] = new_pos
    port.loc[mask,'cost']     = new_cost
    print(f"  ADD      {row['ticker']}/{row['strategy']}  {qty:+d} @ "
          f"{float(row['price_exec']):.4f}  -> pos={new_pos}  cost={new_cost:.4f}")
    return port


def _close_fifo(row, port):
    ticker, strategy = row['ticker'], row['strategy']
    qty_exec   = int(row['quantity_exec'])
    to_close   = abs(qty_exec)
    price_exit = float(row['price_exec'])
    commission = float(row['commission']) if pd.notna(row.get('commission')) else 0.0
    trade_date = row['date']
    open_lots  = at_list_open(ticker, strategy)
    if not open_lots:
        print(f'  WARNING  no open lots for {ticker}/{strategy} -- skipping')
        return port
    remaining = to_close; gross_total = 0.0
    for lot in open_lots:
        if remaining <= 0: break
        lot_id    = lot['id']
        lot_qty   = lot['fields'].get('quantity_exec', 0)
        lot_sign  = 1 if lot_qty > 0 else -1
        lot_abs   = abs(lot_qty)
        lot_price = float(lot['fields'].get('price_exec', 0))
        lot_comm  = float(lot['fields'].get('commission') or 0)
        consumable = min(lot_abs, remaining)
        gross_pnl  = round((price_exit - lot_price) * consumable * lot_sign, 2)
        net_pnl    = round(gross_pnl
                           - lot_comm * consumable / lot_abs
                           - commission * consumable / to_close, 2)
        result = 'win' if net_pnl > 0 else ('loss' if net_pnl < 0 else 'breakeven')
        at_patch(lot_id, {
            'quantity_exec': consumable * lot_sign,
            'exit_date':     trade_date,          # use the execution date, not today()
            'price_exit':    round(price_exit, 4),
            'gross_pnl':     gross_pnl,
            'net_pnl':       net_pnl,
            'trade_result':  result,
        })
        gross_total += gross_pnl; remaining -= consumable
    mask    = (port['ticker'] == ticker) & (port['strategy'] == strategy)
    tot_pos = int(port.loc[mask, 'position'].sum())  # sum over all lots of same position
    new_pos = tot_pos + qty_exec
    if new_pos == 0:
        # ── Full close ─────────────────────────────────────────────────────
        port = port[~mask].reset_index(drop=True)
        print(f'  CLOSE    {ticker}/{strategy}  '
              f'closed={to_close} @ {price_exit:.4f}  gross={gross_total:+.2f}')
    elif remaining > 0:
        # ── Reversal: all prior lots consumed; excess opens new direction ──
        rev_comm = round(commission * remaining / to_close, 2)
        at_append({
            'entry_date':     trade_date,
            'ticker':         ticker,
            'strategy':       strategy,
            'broker':         row.get('broker'),
            'order_type':     row.get('order_type_ord') or row.get('order_type'),
            'quantity_order': int(row.get('quantity', 0) or 0),
            'price_order':    row.get('price'),
            'quantity_exec':  new_pos,
            'price_exec':     round(price_exit, 4),
            'commission':     rev_comm,
            'stop_price':     row.get('stop_price'),
            'target_price':   row.get('target_price'),
            'record_id':      row.get('execution_id'),
        })
        port = port[~mask].reset_index(drop=True)
        new_row = pd.DataFrame([{
            'ticker':       ticker,   'strategy':   strategy,
            'position':     new_pos,
            'cost':         round(price_exit, 4),
            'stop_price':   row.get('stop_price'),
            'target_price': row.get('target_price'),
            'entry_date':   trade_date,
        }])
        port = pd.concat([port, new_row], ignore_index=True)
        closed = to_close - remaining
        print(f'  REVERSAL {ticker}/{strategy}  '
              f'closed={closed} new={new_pos:+d} @ {price_exit:.4f}  gross={gross_total:+.2f}')
    else:
        # ── Partial exit: position reduced but not closed ──────────────────
        first_idx = port[mask].index[0]
        port.loc[first_idx, 'position'] = new_pos
        if mask.sum() > 1:
            port = port.drop(port[mask].index[1:]).reset_index(drop=True)
        print(f'  PARTIAL  {ticker}/{strategy}  '
              f'closed={to_close} remaining={new_pos:+d}  gross={gross_total:+.2f}')
    return port

## 9 · reconcile

The orchestrator. It calls `_split_trades` once to classify all rows,
writes the two exception categories, then loops over matched trades
and routes each one to the correct position handler via a three-way decision:

```
no existing position  ->  _open_new
same direction        ->  _add_lot
opposite direction    ->  _close_fifo
```

Direction is determined by comparing the sign of `quantity_exec` with
the sign of the existing `position`. If they match, we are adding to the
position; if they differ, we are closing or reducing it.

In [ ]:
def reconcile(orders, executions, portfolio):
    matched, missed, override = _split_trades(orders, executions)
    _write_missed(missed)
    _write_override(override)
    port = portfolio.copy().reset_index(drop=True)
    for _, row in matched.iterrows():
        mask = (port['ticker']==row['ticker']) & (port['strategy']==row['strategy'])
        if not mask.any():
            port = _open_new(row, port)
        elif (int(row['quantity_exec'])>0)==(int(port.loc[mask,'position'].values[0])>0):
            port = _add_lot(row, port)
        else:
            port = _close_fifo(row, port)
    return port.reindex(columns=PORTFOLIO_COLS)

## 10 · Run

Pass the three sample DataFrames through `reconcile`, save the updated
portfolio to CSV (overwrites `sample_portfolio.csv`), and display the result.

In production, replace the sample DataFrames with calls to `load_csv`
pointing at your broker export files.

In [ ]:
print('=== Reconciling ===')
updated_portfolio = reconcile(sample_orders, sample_executions, sample_portfolio)
print()
print('=== Updated portfolio ===')
display(updated_portfolio)

updated_portfolio.to_csv('sample_portfolio.csv', index=False)
print('Saved sample_portfolio.csv')

---
# Part B · Psychology Journal Analysis

## What we are building

A trader's edge is not just a strategy — it is also the consistent execution
of that strategy under emotional pressure. Most traders know what to do;
fewer do it every day. The psychology journal bridges that gap.

The journal is split across two Airtable forms to minimise daily friction:

| Form | When | Fields | Time |
|---|---|---|---|
| Morning Check-in | before the open | mindset, confidence, checklist, forecast, reasons | 2 min |
| Evening Review | after the close | reflection, performance, gratitude, kaizen | 5 min |

Each form creates one row in the `journal` table with `session = 'pre'` or `'post'`.
This notebook loads both sessions, joins them by date, and computes four derived fields:

| Field | Formula | Meaning |
|---|---|---|
| `bulls_eye` | sign(perf_forecast x perf_actual) | +1 right call, -1 wrong, 0 neutral |
| `score` | weighted sum of filled fields | completeness reward, max 120 pts |
| `streak` | consecutive weekday entries | habit momentum |
| `multiplier` | 1 + streak / 20 | geometric compounding of the streak |
| `final_score` | score x multiplier | total gamified score |

The computed values are patched back to the `post` row in Airtable
so they are visible in your dashboard without re-running the notebook.

## Steps

11. Load pre and post rows from Airtable (with pagination)
12. Define the four compute functions
13. Merge by date, compute all derived fields
14. Patch computed values back to Airtable
15. Display a summary of the last 10 trading days

## 11 · Load journal rows

`load_journal_session` fetches all rows for one session type, handling
Airtable's pagination automatically (100 records per page by default).

When the table is empty it returns a DataFrame with at least a `date` column
so the merge in Cell 13 does not raise a `KeyError`.

In [ ]:
def load_journal_session(session):
    formula = f"{{session}}='{session}'"
    records, offset = [], None
    while True:
        params = {'filterByFormula': formula,
                  'sort[0][field]': 'date', 'sort[0][direction]': 'asc'}
        if offset: params['offset'] = offset
        r = requests.get(AT_J_URL, headers=AT_HEADERS, params=params)
        r.raise_for_status()
        data     = r.json()
        records += data.get('records', [])
        offset   = data.get('offset')
        if not offset: break
    rows = [{'_id': rec['id'], **rec['fields']} for rec in records]
    # Always include 'date' column so merge works even when table is empty
    return pd.DataFrame(rows) if rows else pd.DataFrame(columns=['_id', 'date'])

pre_df  = load_journal_session('pre')
post_df = load_journal_session('post')
print(f'pre rows : {len(pre_df)}')
print(f'post rows: {len(post_df)}')

if pre_df.empty and post_df.empty:
    print('No entries yet -- submit at least one Morning and one Evening form first.')

## 12 · Compute functions

**`_bulls_eye(pf, pa)`** — compares the sign of your forecast against the
sign of actual performance. Returns `+1` if you called the direction correctly,
`-1` if wrong, `0` if either value is missing or zero. Over time, a positive
average bulls_eye means your directional judgement adds value.

**`_score(row)`** — awards points for completing each field. Text fields
(kaizen, what went well, notes) earn the most because they require effort.
Checklist and kaizen are the two anchors — each worth 20 pts — because they
represent the most important habits: preparation and continuous improvement.

**Score breakdown (max 120 pts):**

| Section | Field | pts |
|---|---|---|
| Pre | `checklist_done = 1` | 20 |
| Pre | `mindset_pre`, `confidence_pre` | 5 each |
| Pre | `perf_forecast` | 5 |
| Pre | `bc_1`, `bc_2` | 3 + 2 |
| Post | `mindset_post`, `confidence_post` | 5 each |
| Post | `perf_actual` | 5 |
| Post | `what_went_well_1/2/3` | 5 each |
| Post | `tomorrows_kaizen` | 20 |
| Post | `notes` | 10 |
| Post | gratitudes: 1 / 2 / all 3 | 6 / 12 / 20 |

**`_add_streaks(df)`** — walks the sorted DataFrame day by day and counts
consecutive trading days. One missed weekday is forgiven (grace period);
two consecutive missed weekdays resets the streak to 1.
The multiplier `1 + streak/20` compounds geometrically: day 20 = x2, day 40 = x3.

In [ ]:
def _bulls_eye(pf, pa):
    if pd.isna(pf) or pd.isna(pa): return None
    p = float(pf) * float(pa)
    return 1 if p > 0 else (-1 if p < 0 else 0)

def _score(row):
    s = 0
    # Pre-market (max 40)
    s += 20 if row.get('checklist_done') == 1  else 0
    s +=  5 if pd.notna(row.get('mindset_pre'))    else 0
    s +=  5 if pd.notna(row.get('confidence_pre')) else 0
    s +=  5 if pd.notna(row.get('perf_forecast'))  else 0
    s +=  3 if str(row.get('bc_1') or '').strip()  else 0
    s +=  2 if str(row.get('bc_2') or '').strip()  else 0
    # Post-market (max 60)
    s +=  5 if pd.notna(row.get('mindset_post'))    else 0
    s +=  5 if pd.notna(row.get('confidence_post')) else 0
    s +=  5 if pd.notna(row.get('perf_actual'))     else 0
    s +=  5 if str(row.get('what_went_well_1') or '').strip() else 0
    s +=  5 if str(row.get('what_went_well_2') or '').strip() else 0
    s +=  5 if str(row.get('what_went_well_3') or '').strip() else 0
    s += 20 if str(row.get('tomorrows_kaizen') or '').strip() else 0
    s += 10 if str(row.get('notes')            or '').strip() else 0
    # Gratitude tiered bonus
    g  = sum(1 for k in ('gratitude_1','gratitude_2','gratitude_3')
             if str(row.get(k) or '').strip())
    s += {0: 0, 1: 6, 2: 12, 3: 20}[g]
    return s   # max 120

def _add_streaks(df):
    df = df.sort_values('date').copy()
    streaks, n, prev = [], 0, None
    for d in pd.to_datetime(df['date']):
        if prev is None:
            n = 1
        else:
            gap = len(pd.bdate_range(prev + pd.Timedelta(days=1), d))
            n   = n + 1 if gap <= 2 else 1
        streaks.append(n)
        prev = d
    df['streak']     = streaks
    df['multiplier'] = (1 + df['streak'] / 20).round(2)
    return df

## 13 · Merge and compute

An outer merge on `date` pairs each morning row with its evening counterpart.
Days with only a pre or only a post entry are kept (`how='outer'`) so nothing
is silently dropped. The `suffixes` parameter handles the handful of field
names that appear in both sessions (e.g. `session`).

`merged` is initialised as an empty DataFrame before the guard so that
Cells 14 and 15 always have the variable defined, even if no data exists yet.

In [ ]:
merged = pd.DataFrame()  # populated below if data exists

if pre_df.empty and post_df.empty:
    print('Nothing to merge yet.')
else:
    merged = pd.merge(
        pre_df.rename(columns={'_id': '_pre_id'}),
        post_df.rename(columns={'_id': '_post_id'}),
        on='date', how='outer', suffixes=('_pre','_post')
    )
    merged['bulls_eye']   = merged.apply(
        lambda r: _bulls_eye(r.get('perf_forecast'), r.get('perf_actual')), axis=1)
    merged['score']       = merged.apply(_score, axis=1)
    merged                = _add_streaks(merged)
    merged['final_score'] = (merged['score'] * merged['multiplier']).round(1)
    print(f'Days merged: {len(merged)}')
    print(merged[['date','score','streak','multiplier','final_score','bulls_eye']]
          .to_string(index=False))

## 14 · Patch computed fields back to Airtable

Writes `bulls_eye`, `score`, `streak`, `multiplier`, and `final_score`
to the **post** row for each day. The pre row is left untouched since
these fields require both morning and evening data to compute.

Days where the post row does not yet exist (only a pre entry submitted)
are skipped with a message. Safe to re-run — patching the same values
twice has no side effects.

In [ ]:
patched = 0
for _, row in merged.iterrows():
    post_id = row.get('_post_id')
    if pd.isna(post_id) if isinstance(post_id, float) else not post_id:
        print(f'  SKIP  {row["date"]}  no post row yet')
        continue
    fields = {}
    for col in ('bulls_eye','score','streak','multiplier','final_score'):
        val = row.get(col)
        if val is not None and not (isinstance(val, float) and pd.isna(val)):
            fields[col] = val
    r = requests.patch(f'{AT_J_URL}/{post_id}', headers=AT_HEADERS,
                       json={'fields': fields})
    r.raise_for_status()
    patched += 1
print(f'Patched {patched} post rows.')

## 15 · Summary

The last 10 trading days sorted most-recent first.
Only columns that exist in the merged DataFrame are shown,
so the cell runs cleanly even with partial data.

In [ ]:
COLS  = ['date','streak','multiplier','score','final_score',
         'bulls_eye','mindset_pre','mindset_post','perf_forecast','perf_actual']
if merged.empty:
    print('No journal data yet — submit at least one Morning and one Evening form first.')
else:
    avail = [c for c in COLS if c in merged.columns]
    display(merged.sort_values('date', ascending=False)[avail].head(10))